## Setup

In [15]:
import logging
import os
from pathlib import Path
import re
import subprocess

import pandas as pd

# Custom words to remove
EXCLUDE_LIST = ['[FREE DL]', '[DUPLOC10YRS]', '[Digital Bonus]', '[Premiere]', '[Bassrush Premiere]', '(DDD118)']
REPLACE_DICT = {
    "¿": ""
}


def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df


def strip_digit_prefix(stem):
    # "2.Sub Basics - Shipment Pt.2" => "Sub Basics - Shipment Pt.2"
    return re.sub(r'^\d+\.', '', stem)


def strip_underscore_digits(stem):
    # "Criso & Breathworkz - Wicked_2051092604 - Criso" => "Criso & Breathworkz - Wicked - Criso"
    return re.sub(r'_\d+', '', stem)


def remove_words(series):
    replace = REPLACE_DICT
    remove = {s: "" for s in EXCLUDE_LIST}
    replace.update(remove)
    series = series.str.replace(replace)
    # Remove consecutive whitespace and strip
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()
    return series


def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logging.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)


## Run

In [59]:
src_dir = Path("C:/Users/sesti/Music/Playlists/fresh/sadgirl")
dst_dir = Path("C:/Users/sesti/Music/Playlists/sadgirl/")

In [60]:
! ls $src_dir

1.Midwife & Vyva Melinkolya - Miss America.mp3
2.Midwife - Autoluminescent.mp3
3.Midwife - 2020.mp3
4.Midwife - Colorado.mp3
5.Midwife - Killdozer.mp3


In [61]:
! mkdir $dst_dir & ls $dst_dir

In [62]:
df = get_file_df(src_dir)
df["new_stem"] = df.stem.map(strip_digit_prefix).map(strip_underscore_digits)
df["new_stem"] = remove_words(df.new_stem).str.strip()
df["new_suffix"] = ".wav"
df["new_parent"] = dst_dir
df["new_sample_rate"] = str(44100)
df["new_sample_fmt"] = "s16"
df["new_path"] = df.apply(lambda row: row.new_parent / (row.new_stem + row.new_suffix), axis=1).values
df["exists"] = df["new_path"].apply(os.path.exists)

df[['new_stem', 'exists', 'new_path']]

,new_stem,exists,new_path
0,Midwife & Vyva Melinkolya - Miss America,False,C:\Users\sesti\Music\Playlists\sadgirl\Midwife...
1,Midwife - Autoluminescent,False,C:\Users\sesti\Music\Playlists\sadgirl\Midwife...
2,Midwife - 2020,False,C:\Users\sesti\Music\Playlists\sadgirl\Midwife...
3,Midwife - Colorado,False,C:\Users\sesti\Music\Playlists\sadgirl\Midwife...
4,Midwife - Killdozer,False,C:\Users\sesti\Music\Playlists\sadgirl\Midwife...


In [63]:
df.apply(run_ffmpeg, axis=1)

0    0
1    0
2    0
3    0
4    0
dtype: int64

In [64]:
! ls $dst_dir

Midwife - 2020.wav
Midwife - Autoluminescent.wav
Midwife - Colorado.wav
Midwife - Killdozer.wav
Midwife & Vyva Melinkolya - Miss America.wav
